# 03 · Gold · `fato_consumo_faturamento`
### V-Energy Grid 360 — Arquitetura Medalhão

**Status: executada no Databricks, com todas as checagens `PASS`.** Reproduz o diagnóstico: volume −3,95% e receita de energia +8,00%, e a variação de volume por classe confere. A conexão com a `gold.dim_uc` ainda não foi testada.

Une **medição e faturamento**, que hoje vivem em sistemas separados, numa tabela única por UC e mês. Grava `v_energy.gold.fato_consumo_faturamento`.

**Grão:** 1 linha = 1 unidade consumidora em 1 mês de referência (1 leitura + 1 fatura). Esperado: **2.152.336 linhas**, com `(id_uc, mes_referencia)` único.

**Quem usa:** visão 360 (gráfico de consumo contra o mesmo mês do ano anterior e lista de faturas em aberto), agente de IA e diagnóstico da receita.

**Fontes (Silver):** `leituras`, `faturas`, `unidades_consumidoras` e `geracao_distribuida`. Esta tabela **não depende** da `dim_uc` da Gold, então pode ser construída em paralelo.

| # | Passo | Regra |
|---|---|---|
| 1 | Leitura + fatura | `left join` de `silver.leituras` com `silver.faturas` por `(id_uc, mes_referencia)`. A Silver garante 1 fatura por leitura |
| 2 | Atributos da UC e GD | `classe`, `bairro` e `municipio` copiados de `silver.unidades_consumidoras`; `possui_gd_no_mes` a partir da data de conexão em `silver.geracao_distribuida` |
| 3 | `ano`, `mes`, `periodo_diagnostico` | derivados de `mes_referencia`; períodos do diagnóstico: primeiro ano = `2022-07` a `2023-06`, últimos 12 meses = `2025-07` a `2026-06` |
| 4 | Ano anterior | `consumo_kwh_ano_anterior` e `variacao_consumo_ano_anterior_pct`, por junção da mesma UC no mesmo mês do ano anterior |
| 5 | Atraso | `dias_atraso` (0 se pago em dia; até `2026-06-30` se em aberto) e `faixa_atraso` (`EM DIA`, `ATE 30 DIAS`, `31 A 90 DIAS`, `MAIS DE 90 DIAS`) |
| 6 | Preço | `preco_medio_kwh = valor_energia / consumo_kwh` |

**Por que a tabela passa por uma tabela temporária (`_stg`):** o Spark serverless não permite `.cache()`, e cada checagem recalcularia as junções inteiras (2 milhões de linhas e uma junção da tabela com ela mesma). O resultado é gravado primeiro em `gold.fato_consumo_faturamento_stg`; as checagens leem dela; **só se todas passarem** a tabela final é gravada e a temporária é apagada. Se alguma falhar, a final fica como estava e a `_stg` permanece para inspeção.

**O que entra no dicionário de dados desta tabela** (decisões da Silver que afetam os números):
- `consumo_kwh` nos meses de troca de medidor (`troca_medidor = true`, 6.299 linhas) é a `leitura_atual` (log L5), e `preco_medio_kwh` nesses meses fica ~5% acima do típico;
- `consumo_kwh` com `consumo_corrigido = true` (1.717 linhas) teve o sinal corrigido (L4);
- 32.285 faturas e 8.609 leituras órfãs foram removidas na Silver (F2, L2), então a receita total é ~1,5% menor que a da Bronze;
- ~8% das faturas ficam "em aberto" **em todos os anos, até 2022**: isso afeta `dias_atraso` e qualquer indicador de inadimplência;
- `consumo_kwh` já parece vir **líquido da geração distribuída** (o consumo faturado cai ~69% depois da conexão): `possui_gd_no_mes` marca os meses em que a UC já tinha GD conectada, **incluindo o mês da conexão**, que costuma cair só em parte;
- a inadimplência **não** deve ser calculada com `dias_atraso` de faturas pagas (83% delas são pagas com algum atraso; o atraso médio, contando também as pagas em dia, é 10,8 dias, e o máximo é 25): o contrato define inadimplência por competência, como % do valor em aberto.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_leituras = f"{silver_schema}.leituras"
tabela_faturas = f"{silver_schema}.faturas"
tabela_uc = f"{silver_schema}.unidades_consumidoras"
tabela_gd = f"{silver_schema}.geracao_distribuida"
tabela_stg = f"{gold_schema}.fato_consumo_faturamento_stg"
tabela_gold = f"{gold_schema}.fato_consumo_faturamento"
tabela_dq_log = f"{silver_schema}.dq_log"      # mesmo log das Silvers, mesmas colunas

# Períodos do diagnóstico (mes_referencia como texto yyyy-MM, que ordena corretamente)
P0_INI, P0_FIM = "2022-07", "2023-06"          # primeiro ano
P1_INI, P1_FIM = "2025-07", "2026-06"          # últimos 12 meses
DATA_FIM_BASE = "2026-06-30"                   # fim da base: referência de atraso das faturas em aberto

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"origem:  {tabela_leituras}, {tabela_faturas}, {tabela_uc}, {tabela_gd}")
print(f"destino: {tabela_gold} (via {tabela_stg})")

origem:  v_energy.silver.leituras, v_energy.silver.faturas, v_energy.silver.unidades_consumidoras, v_energy.silver.geracao_distribuida
destino: v_energy.gold.fato_consumo_faturamento (via v_energy.gold.fato_consumo_faturamento_stg)


## 2. Funções de checagem de qualidade

Mesmas funções da aula e das Silvers (`dq_check`, `dq_check_unique` e `dq_check_contagem`), com as mesmas colunas no log (`table_name`, `check_name`, `total_rows`, `failed_rows`, `passed`, `checked_at`). Acrescento `dq_check_valor`, para comparar um número calculado com um esperado (somas que precisam bater com as Silvers e com o diagnóstico); ela grava no log nas mesmas colunas, com `failed_rows` igual a 0 ou 1.

As funções só acumulam os resultados em `dq_results`. Depois das checagens, o notebook **grava o resultado em `silver.dq_log`** (sempre) e **só então decide se grava a Gold**.

**Cuidado com nulos:** linhas em que a condição resulta em nulo não contam como falha. Quando o nulo também for um problema, há um `dq_check` separado com `isNotNull()`.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

def dq_check_valor(table_name: str, check_name: str, total_rows: int, obtido, esperado):
    """Compara um valor calculado com o esperado (soma que precisa bater com a Silver ou com o diagnóstico)."""
    passed = obtido == esperado
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total_rows,
            failed_rows=0 if passed else 1, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | obtido {obtido} (esperado {esperado})")

## 3. Leitura das Silvers e conferência das colunas

Antes de construir, o notebook confere que as três Silvers existem **com as colunas de que precisa**. Se alguma estiver faltando (por exemplo, se uma Silver mudou de nome de coluna), ele para aqui com uma mensagem clara, e não no meio de uma junção.

Contagens esperadas:
- `silver.geracao_distribuida`: **3.510 linhas e 3.510 UCs distintas** (1 linha por UC);
- `silver.leituras`: **2.152.336** (uma por UC e mês);
- `silver.faturas`: **2.152.336** (uma por leitura);
- `silver.unidades_consumidoras`: **45.000** (uma por UC).

In [0]:
leit = spark.table(tabela_leituras)
fat = spark.table(tabela_faturas)
uc = spark.table(tabela_uc)
gd = spark.table(tabela_gd)

exigidas = {
    tabela_leituras: ["id_uc", "mes_referencia", "id_medidor", "data_leitura", "tipo_leitura", "consumo_kwh",
                      "troca_medidor", "consumo_corrigido", "leitura_anterior_ausente"],
    tabela_faturas: ["id_fatura", "id_uc", "mes_referencia", "data_emissao", "data_vencimento", "data_pagamento",
                     "valor_energia", "tributos", "valor_total", "bandeira_tarifaria", "status", "em_aberto"],
    tabela_uc: ["id_uc", "classe", "bairro", "municipio"],
    tabela_gd: ["id_uc", "data_conexao"],
}
for tabela, colunas in exigidas.items():
    faltando = [c for c in colunas if c not in spark.table(tabela).columns]
    if faltando:
        raise Exception(f"{tabela} não tem as colunas {faltando}. Ajuste os nomes antes de seguir (DESCRIBE {tabela}).")

n_leit, n_fat, n_uc = leit.count(), fat.count(), uc.count()
print(f"leituras: {n_br(n_leit)} | faturas: {n_br(n_fat)} | UCs: {n_br(n_uc)}")   # esperado: 2.152.336 | 2.152.336 | 45.000
print("UCs distintas no cadastro:", n_br(uc.select("id_uc").distinct().count()))    # esperado: 45.000 (id_uc único)
print("GD:", n_br(gd.count()), "linhas |", n_br(gd.select("id_uc").distinct().count()), "UCs distintas")   # esperado: 3.510 UCs (1 linha por UC)
leit.printSchema()

leituras: 2.152.336 | faturas: 2.152.336 | UCs: 45.000
UCs distintas no cadastro: 45.000
GD: 3.510 linhas | 3.510 UCs distintas
root
 |-- id_leitura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- id_medidor: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- data_leitura: date (nullable = true)
 |-- leitura_anterior: long (nullable = true)
 |-- leitura_atual: long (nullable = true)
 |-- consumo_kwh: long (nullable = true)
 |-- consumo_kwh_original: long (nullable = true)
 |-- tipo_leitura: string (nullable = true)
 |-- leitura_anterior_ausente: boolean (nullable = true)
 |-- consumo_corrigido: boolean (nullable = true)
 |-- troca_medidor: boolean (nullable = true)
 |-- source_file: string (nullable = true)



## 4. Leitura + fatura + atributos da UC

**Decisão:** `left join` de `silver.leituras` com `silver.faturas` pela chave `(id_uc, mes_referencia)`, e depois com `silver.unidades_consumidoras` por `id_uc`, trazendo `classe`, `bairro` e `municipio`.

**Evidência:** a Silver de `faturas` verifica que as duas tabelas têm exatamente os mesmos 2.152.336 pares `(id_uc, mes_referencia)` (0 faturas sem leitura e 0 leituras sem fatura), e a chave é única nas duas. A junção, portanto, não multiplica nem perde linhas. A Silver de UCs tem 45.000 UCs distintas, e toda UC com leitura existe nela.

**Por que `left` a partir de `leituras`:** se uma fatura ou uma UC faltasse, a linha continuaria na tabela com a coluna nula, e as checagens da seção 8 acusariam, em vez de a linha sumir em silêncio (como aconteceria num `inner`).

**Por que os atributos da UC se repetem aqui:** o contrato pede `classe`, `bairro` e `municipio` nesta tabela para os filtros do portal e as consultas do agente. A fonte de verdade continua sendo a `dim_uc`. O bairro só tem sentido junto do município (o mesmo nome existe em municípios diferentes).

**GD no mês (`possui_gd_no_mes`):** a data de conexão vem de `silver.geracao_distribuida` (uma data por UC; se houvesse mais de uma linha por UC, vale a mais antiga). O valor é `true` quando `mes_referencia` é **igual ou posterior** ao mês da conexão, que é a regra "GD conectada até o mês" do contrato. Sem GD, `false`. A regra precisa ser **a mesma** da `qtd_ucs_com_gd` da `fato_segmento_mensal` e da `kpis_mensal` (Aimê e eu).

**Colunas:** só as do contrato. Ficam de fora `id_leitura`, `leitura_anterior`, `leitura_atual`, `consumo_kwh_original`, `source_file` e a coluna `status` da fatura, que vira `status_fatura`.

**Conferência:** 2.152.336 linhas depois das três junções, 0 sem fatura e 0 sem classe, bairro ou município.

In [0]:
# Junções (esperado: 2.152.336 linhas; 0 sem fatura; 0 sem classe/bairro/município)
leit_sel = leit.select(
    "id_uc", "mes_referencia", "id_medidor", "data_leitura", "tipo_leitura", "consumo_kwh",
    "troca_medidor", "consumo_corrigido", "leitura_anterior_ausente",
)
fat_sel = fat.select(
    "id_uc", "mes_referencia", "id_fatura", "data_emissao", "data_vencimento", "data_pagamento",
    "valor_energia", "tributos", "valor_total", "bandeira_tarifaria",
    F.col("status").alias("status_fatura"), "em_aberto",
)
uc_sel = uc.select("id_uc", "classe", "bairro", "municipio")
gd_sel = gd.groupBy("id_uc").agg(F.min("data_conexao").alias("data_conexao_gd"))   # 1 linha por UC

base = (
    leit_sel
    .join(fat_sel, ["id_uc", "mes_referencia"], "left")
    .join(F.broadcast(uc_sel), "id_uc", "left")
    .join(F.broadcast(gd_sel), "id_uc", "left")
    .withColumn("possui_gd_no_mes",
                F.coalesce(F.col("mes_referencia") >= F.date_format(F.col("data_conexao_gd"), "yyyy-MM"), F.lit(False)))
)

n_base = base.count()
print(f"Linhas depois das junções: {n_br(n_base)}")                                               # esperado: 2.152.336
print("sem fatura:", n_br(base.filter("id_fatura IS NULL").count()))                               # esperado: 0
print("UCs com GD em algum mês:", n_br(base.filter("possui_gd_no_mes").select("id_uc").distinct().count()))   # esperado: até 3.510 (3.510 se todas as conexões forem até 2026-06)
print("sem classe/bairro/município:",
      n_br(base.filter("classe IS NULL OR bairro IS NULL OR municipio IS NULL").count()))           # esperado: 0

Linhas depois das junções: 2.152.336
sem fatura: 0
UCs com GD em algum mês: 3.510
sem classe/bairro/município: 0


## 5. `ano`, `mes` e `periodo_diagnostico`

**Decisão:** `ano` e `mes` saem de `mes_referencia` (texto `yyyy-MM`), com `try_cast`. `periodo_diagnostico` classifica cada mês:

| Valor | Meses | Quantos |
|---|---|---|
| `PRIMEIRO_ANO` | `2022-07` a `2023-06` | 12 |
| `ULTIMOS_12M` | `2025-07` a `2026-06` | 12 |
| `OUTRO` | os demais (`2023-07` a `2025-06`) | 24 |

**Por que a coluna existe:** a pergunta do diretor compara exatamente esses dois períodos. Com a coluna, a tela, o agente e o diagnóstico usam a **mesma definição**, e ninguém reescreve o intervalo em cada consulta.

**Cuidado:** a comparação de `mes_referencia` como texto funciona porque o formato `yyyy-MM` ordena corretamente.

**Conferência:** nenhum `ano` ou `mes` nulo, e os três valores de `periodo_diagnostico` com 12, 12 e 24 meses distintos.

In [0]:
# ano, mes e periodo_diagnostico (esperado: 0 nulos; PRIMEIRO_ANO = 12 meses, ULTIMOS_12M = 12, OUTRO = 24)
base2 = (
    base
    .withColumn("ano", F.expr("try_cast(substring(mes_referencia, 1, 4) as int)"))
    .withColumn("mes", F.expr("try_cast(substring(mes_referencia, 6, 2) as int)"))
    .withColumn(
        "periodo_diagnostico",
        F.when(F.col("mes_referencia").between(P0_INI, P0_FIM), "PRIMEIRO_ANO")
         .when(F.col("mes_referencia").between(P1_INI, P1_FIM), "ULTIMOS_12M")
         .otherwise("OUTRO"),
    )
)
print("ano ou mes nulo:", n_br(base2.filter("ano IS NULL OR mes IS NULL").count()))              # esperado: 0
(base2.groupBy("periodo_diagnostico")
      .agg(F.countDistinct("mes_referencia").alias("meses"), F.count("*").alias("linhas"))
      .orderBy("periodo_diagnostico").show())

ano ou mes nulo: 0
+-------------------+-----+-------+
|periodo_diagnostico|meses| linhas|
+-------------------+-----+-------+
|              OUTRO|   24|1076301|
|       PRIMEIRO_ANO|   12| 536267|
|        ULTIMOS_12M|   12| 539768|
+-------------------+-----+-------+



## 6. Consumo do mesmo mês do ano anterior e variação

**Decisão:** para cada UC e mês, buscar o consumo da **mesma UC no mesmo mês do ano anterior**, por uma junção da tabela com ela mesma (a chave é `(id_uc, mês + 12 meses)`). A variação é `(consumo − consumo_ano_anterior) / consumo_ano_anterior × 100`, com 2 casas.

**Por que junção e não `lag(12)`:** a base tem UCs que entram depois de 2022-07 (a base cresce), então o histórico de uma UC pode ter meses faltando no começo. Um `lag(12)` por posição pegaria o mês errado nesses casos, sem dar erro. A junção pelo mês exato nunca erra: se o mês do ano anterior não existe, o valor fica nulo.

**Nulos esperados (nulo de negócio, preservado):**
- **todos os meses de `2022-07` a `2023-06`** (o primeiro ano da base não tem ano anterior);
- meses seguintes de UCs que ainda não existiam no mesmo mês do ano anterior.

`try_divide` devolve nulo se o consumo do ano anterior for zero ou nulo, em vez de dar erro (o modo ANSI do serverless interrompe a divisão por zero).

**Atenção para quem usa:** nos meses de troca de medidor, o `consumo_kwh` é a `leitura_atual` (L5). A comparação com o ano anterior continua válida, mas a flag `troca_medidor` avisa que o mês tem um critério diferente. É a informação que o atendente precisa quando o cliente diz que "a conta veio alta".

**Conferência:** a quantidade de linhas não muda (a tabela do ano anterior tem uma linha por UC e mês), e nenhum mês de `2022-07` a `2023-06` tem valor.

In [0]:
# Ano anterior (esperado: mesma contagem de linhas; 0 valores nos meses 2022-07 a 2023-06)
ano_ant = base2.select(
    "id_uc",
    F.concat((F.col("ano") + 1).cast("string"), F.lit("-"), F.lpad(F.col("mes").cast("string"), 2, "0")).alias("mes_referencia"),
    F.col("consumo_kwh").alias("consumo_kwh_ano_anterior"),
)

base3 = (
    base2.join(ano_ant, ["id_uc", "mes_referencia"], "left")
    .withColumn(
        "variacao_consumo_ano_anterior_pct",
        F.round(F.expr("try_divide(consumo_kwh - consumo_kwh_ano_anterior, consumo_kwh_ano_anterior) * 100"), 2).cast("decimal(10,2)"),
    )
)

n_base3 = base3.count()
print(f"Linhas depois da junção com o ano anterior: {n_br(n_base3)} (antes: {n_br(n_base)})")   # esperado: igual
print("com consumo do ano anterior em 2022-07 a 2023-06:",
      n_br(base3.filter((F.col("mes_referencia") <= P0_FIM) & F.col("consumo_kwh_ano_anterior").isNotNull()).count()))   # esperado: 0
print("sem consumo do ano anterior a partir de 2023-07:",
      n_br(base3.filter((F.col("mes_referencia") > P0_FIM) & F.col("consumo_kwh_ano_anterior").isNull()).count()))        # informativo: UCs novas

Linhas depois da junção com o ano anterior: 2.152.336 (antes: 2.152.336)
com consumo do ano anterior em 2022-07 a 2023-06: 0
sem consumo do ano anterior a partir de 2023-07: 3.501


## 7. `dias_atraso`, `preco_medio_kwh` e seleção final

### `dias_atraso`
**Decisão:** dias entre o **vencimento** e o **pagamento**; se a fatura está **em aberto**, até **2026-06-30** (fim da base). Nunca negativo: pago antes do vencimento ou em aberto antes do vencimento vale 0.

**Cuidado:** na Silver, ~8% das faturas ficam `EM ABERTO` **em todos os anos, até em 2022**. É um traço da base e não inadimplência recente. Para essas faturas, `dias_atraso` chega a mais de mil dias. A lista de "faturas em aberto" da visão 360 deve considerar isso (por exemplo, destacar as mais recentes), e o dicionário de dados deve registrar. A decisão de como mostrar é do Dev e da Camila; a tabela guarda todas, com a flag `em_aberto`.

### `faixa_atraso`
**Decisão:** classificação de `dias_atraso`, pedida pelo contrato para a visão 360 e para a inadimplência por faixa:

| Faixa | `dias_atraso` |
|---|---|
| `EM DIA` | 0 |
| `ATE 30 DIAS` | 1 a 30 |
| `31 A 90 DIAS` | 31 a 90 |
| `MAIS DE 90 DIAS` | mais de 90 |

É a coluna que permite ao Dev separar, na lista de faturas em aberto, as recentes das antigas, sem calcular nada.

### `preco_medio_kwh`
**Decisão:** `valor_energia / consumo_kwh`, com 4 casas (`decimal(12,4)`). Nulo se o consumo for zero (a Silver não tem consumo zero). Usa a energia **sem tributos**, como o diagnóstico (a variação é a mesma com `valor_total`: +8,00% nos dois).

**Cuidado:** nos meses de troca de medidor, o preço fica ~5% acima do típico (o consumo é a `leitura_atual`; log L5).

### Seleção final
As **31 colunas** na ordem do contrato v3. `ano`, `mes` e `periodo_diagnostico` ficam junto de `mes_referencia`.

In [0]:
# dias_atraso, faixa_atraso, preco_medio_kwh e seleção final (31 colunas, na ordem do contrato v3)
referencia_atraso = F.when(F.col("em_aberto"), F.lit(DATA_FIM_BASE).cast("date")).otherwise(F.col("data_pagamento"))

df_fato = (
    base3
    .withColumn("dias_atraso", F.greatest(F.datediff(referencia_atraso, F.col("data_vencimento")), F.lit(0)))
    .withColumn(
        "faixa_atraso",
        F.when(F.col("dias_atraso") == 0, "EM DIA")
         .when(F.col("dias_atraso") <= 30, "ATE 30 DIAS")
         .when(F.col("dias_atraso") <= 90, "31 A 90 DIAS")
         .otherwise("MAIS DE 90 DIAS"),
    )
    .withColumn("preco_medio_kwh", F.round(F.expr("try_divide(valor_energia, consumo_kwh)"), 4).cast("decimal(12,4)"))
    .select(
        "id_uc", "mes_referencia", "ano", "mes", "periodo_diagnostico",
        "classe", "bairro", "municipio", "possui_gd_no_mes",
        "id_medidor", "data_leitura", "tipo_leitura",
        "consumo_kwh", "consumo_kwh_ano_anterior", "variacao_consumo_ano_anterior_pct",
        "troca_medidor", "consumo_corrigido", "leitura_anterior_ausente",
        "id_fatura", "data_emissao", "data_vencimento", "data_pagamento",
        "valor_energia", "tributos", "valor_total",
        "bandeira_tarifaria", "status_fatura", "em_aberto",
        "dias_atraso", "faixa_atraso", "preco_medio_kwh",
    )
)
print("colunas:", len(df_fato.columns))   # esperado: 31
df_fato.printSchema()

# Grava na tabela temporária; as checagens leem dela (sem recalcular as junções)
(
    df_fato.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_stg)
)
fato = spark.table(tabela_stg)
print(f"{tabela_stg}: {n_br(fato.count())} linhas")   # esperado: 2.152.336

colunas: 31
root
 |-- id_uc: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- periodo_diagnostico: string (nullable = false)
 |-- classe: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- possui_gd_no_mes: boolean (nullable = false)
 |-- id_medidor: string (nullable = true)
 |-- data_leitura: date (nullable = true)
 |-- tipo_leitura: string (nullable = true)
 |-- consumo_kwh: long (nullable = true)
 |-- consumo_kwh_ano_anterior: long (nullable = true)
 |-- variacao_consumo_ano_anterior_pct: decimal(10,2) (nullable = true)
 |-- troca_medidor: boolean (nullable = true)
 |-- consumo_corrigido: boolean (nullable = true)
 |-- leitura_anterior_ausente: boolean (nullable = true)
 |-- id_fatura: string (nullable = true)
 |-- data_emissao: date (nullable = true)
 |-- data_vencimento: date (nullable = true)
 |-- data_pagamento: date (nullab

## 8. Checagens de qualidade

Rodam sobre a tabela temporária, **antes** de gravar a tabela final. Cada uma protege uma decisão acima:

| Checagem | Protege |
|---|---|
| Total de linhas = 2.152.336; `(id_uc, mes_referencia)` único | junções que perdem ou multiplicam linhas |
| Colunas-chave não nulas (fatura, leitura, classe, bairro, município, valores, flags) | junções sem correspondência |
| `id_fatura` = `FT` + 9 dígitos; `id_medidor` = `MD` + 6 dígitos | formato das chaves |
| `possui_gd_no_mes` nunca volta de `true` para `false` numa UC; `faixa_atraso` nas 4 faixas e coerente com `dias_atraso` | GD no mês e faixas |
| `classe` nas 5 classes; `bandeira_tarifaria` nas 4 bandeiras; `status_fatura` em `PAGA` ou `EM ABERTO` | domínio das categorias |
| `periodo_diagnostico` nos 3 valores; `ano` e `mes` coerentes com `mes_referencia` | derivados do mês |
| Sem consumo do ano anterior em `2022-07` a `2023-06`; variação preenchida sempre que há ano anterior | junção do ano anterior |
| `consumo_kwh` e valores >= 0; `valor_total = valor_energia + tributos` | tratamentos das Silvers |
| `em_aberto` = ausência de `data_pagamento`; `dias_atraso` >= 0 | atraso |
| **Soma de `consumo_kwh` e de `valor_energia` = soma das Silvers** | a Gold não perdeu nem multiplicou nada |
| **Volume dos dois períodos = o do diagnóstico** (227.052.927 e 218.079.054 kWh) | a Gold reproduz os números que o diretor viu |

As duas últimas são as mais importantes: provam que a tabela fecha com as Silvers e com o diagnóstico (`docs/diagnostico_receita.md`). Se a segunda falhar, não é erro da Gold por si só: é um sinal de que a Silver mudou depois do diagnóstico, e vale entender por quê antes de seguir.

As contagens e os volumes esperados valem para esta extração; numa reexecução com dados novos, precisam ser atualizados.

In [0]:
t = "gold.fato_consumo_faturamento"
dq_results.clear()   # se esta célula for reexecutada, não acumula resultados antigos

# Volume e chave
dq_check_contagem(t, "total de linhas = 2.152.336", fato, 2_152_336)
dq_check_unique(t, "(id_uc, mes_referencia) único", fato, ["id_uc", "mes_referencia"])

# Nulos (data_pagamento, consumo_kwh_ano_anterior e variacao_... são nulos de negócio)
for coluna in ["id_uc", "mes_referencia", "ano", "mes", "periodo_diagnostico", "classe", "bairro", "municipio",
               "id_medidor", "data_leitura", "tipo_leitura", "consumo_kwh",
               "troca_medidor", "consumo_corrigido", "leitura_anterior_ausente",
               "id_fatura", "data_emissao", "data_vencimento", "valor_energia", "tributos", "valor_total",
               "bandeira_tarifaria", "status_fatura", "em_aberto", "dias_atraso", "faixa_atraso", "preco_medio_kwh",
               "possui_gd_no_mes"]:
    dq_check(t, f"{coluna} não nulo", fato, F.col(coluna).isNotNull())

# Formato das chaves e domínios
dq_check(t, "id_fatura no padrão FT + 9 dígitos", fato, F.col("id_fatura").rlike("^FT[0-9]{9}$"))
dq_check(t, "id_medidor no padrão MD + 6 dígitos", fato, F.col("id_medidor").rlike("^MD[0-9]{6}$"))
dq_check(t, "classe só nas 5 classes", fato,
         F.coalesce(F.col("classe").isin("RESIDENCIAL", "RURAL", "COMERCIAL", "PODER PUBLICO", "INDUSTRIAL"), F.lit(False)))
dq_check(t, "bandeira_tarifaria só nas 4 bandeiras", fato,
         F.coalesce(F.col("bandeira_tarifaria").isin("Verde", "Amarela", "Vermelha P1", "Vermelha P2"), F.lit(False)))
dq_check(t, "status_fatura só PAGA ou EM ABERTO", fato,
         F.coalesce(F.col("status_fatura").isin("PAGA", "EM ABERTO"), F.lit(False)))
dq_check(t, "periodo_diagnostico só nos 3 valores", fato,
         F.coalesce(F.col("periodo_diagnostico").isin("PRIMEIRO_ANO", "ULTIMOS_12M", "OUTRO"), F.lit(False)))

# GD no mês e faixa de atraso
w_gd = Window.partitionBy("id_uc").orderBy("mes_referencia")
gd_seq = fato.withColumn("gd_mes_anterior", F.lag("possui_gd_no_mes").over(w_gd))
dq_check(t, "possui_gd_no_mes nunca volta de true para false numa UC", gd_seq,
         ~(F.col("gd_mes_anterior") & ~F.col("possui_gd_no_mes")))
dq_check(t, "faixa_atraso só nas 4 faixas", fato,
         F.coalesce(F.col("faixa_atraso").isin("EM DIA", "ATE 30 DIAS", "31 A 90 DIAS", "MAIS DE 90 DIAS"), F.lit(False)))
dq_check(t, "faixa_atraso coerente com dias_atraso", fato,
         ((F.col("faixa_atraso") == "EM DIA") & (F.col("dias_atraso") == 0))
         | ((F.col("faixa_atraso") == "ATE 30 DIAS") & F.col("dias_atraso").between(1, 30))
         | ((F.col("faixa_atraso") == "31 A 90 DIAS") & F.col("dias_atraso").between(31, 90))
         | ((F.col("faixa_atraso") == "MAIS DE 90 DIAS") & (F.col("dias_atraso") > 90)))

# Derivados do mês
dq_check(t, "ano e mes coerentes com mes_referencia", fato,
         F.col("mes_referencia") == F.concat(F.col("ano").cast("string"), F.lit("-"), F.lpad(F.col("mes").cast("string"), 2, "0")))
dq_check(t, "mes entre 1 e 12", fato, F.col("mes").between(1, 12))

# Ano anterior
dq_check(t, "sem consumo do ano anterior em 2022-07 a 2023-06", fato,
         (F.col("mes_referencia") > P0_FIM) | F.col("consumo_kwh_ano_anterior").isNull())
dq_check(t, "variação preenchida sempre que há consumo do ano anterior", fato,
         F.col("consumo_kwh_ano_anterior").isNull() | F.col("variacao_consumo_ano_anterior_pct").isNotNull())

# Valores
dq_check(t, "consumo_kwh >= 0", fato, F.col("consumo_kwh") >= 0)
dq_check(t, "valor_energia, tributos e valor_total >= 0", fato,
         (F.col("valor_energia") >= 0) & (F.col("tributos") >= 0) & (F.col("valor_total") >= 0))
dq_check(t, "valor_total = valor_energia + tributos", fato,
         F.col("valor_total") == F.col("valor_energia") + F.col("tributos"))

# Fatura em aberto e atraso
dq_check(t, "em_aberto = data_pagamento nula", fato, F.col("em_aberto") == F.col("data_pagamento").isNull())
dq_check(t, "dias_atraso >= 0", fato, F.col("dias_atraso") >= 0)

# Reconciliação com as Silvers: a Gold não pode perder nem multiplicar nada
n_fato = fato.count()
soma_fato = fato.agg(F.sum("consumo_kwh").alias("kwh"), F.sum("valor_energia").alias("rec")).first()
soma_leit = leit.agg(F.sum("consumo_kwh")).first()[0]
soma_fat = fat.agg(F.sum("valor_energia")).first()[0]
dq_check_valor(t, "soma de consumo_kwh = soma de silver.leituras", n_fato, soma_fato["kwh"], soma_leit)
dq_check_valor(t, "soma de valor_energia = soma de silver.faturas", n_fato, soma_fato["rec"], soma_fat)

# Reconciliação com o diagnóstico (docs/diagnostico_receita.md): volume dos dois períodos
vol = {r["periodo_diagnostico"]: r["kwh"] for r in
       fato.filter(F.col("periodo_diagnostico") != "OUTRO")
           .groupBy("periodo_diagnostico").agg(F.sum("consumo_kwh").alias("kwh")).collect()}
dq_check_valor(t, "volume do primeiro ano = 227.052.927 kWh (diagnóstico)", n_fato, vol.get("PRIMEIRO_ANO"), 227_052_927)
dq_check_valor(t, "volume dos últimos 12 meses = 218.079.054 kWh (diagnóstico)", n_fato, vol.get("ULTIMOS_12M"), 218_079_054)

[PASS] gold.fato_consumo_faturamento | total de linhas = 2.152.336 | 2.152.336 linhas (esperado 2.152.336)
[PASS] gold.fato_consumo_faturamento | (id_uc, mes_referencia) único | 0 chaves duplicadas de 2152336 linhas
[PASS] gold.fato_consumo_faturamento | id_uc não nulo | 0/2152336 linhas falharam
[PASS] gold.fato_consumo_faturamento | mes_referencia não nulo | 0/2152336 linhas falharam
[PASS] gold.fato_consumo_faturamento | ano não nulo | 0/2152336 linhas falharam
[PASS] gold.fato_consumo_faturamento | mes não nulo | 0/2152336 linhas falharam
[PASS] gold.fato_consumo_faturamento | periodo_diagnostico não nulo | 0/2152336 linhas falharam
[PASS] gold.fato_consumo_faturamento | classe não nulo | 0/2152336 linhas falharam
[PASS] gold.fato_consumo_faturamento | bairro não nulo | 0/2152336 linhas falharam
[PASS] gold.fato_consumo_faturamento | municipio não nulo | 0/2152336 linhas falharam
[PASS] gold.fato_consumo_faturamento | id_medidor não nulo | 0/2152336 linhas falharam
[PASS] gold.fato

## 9. Registro das checagens em `silver.dq_log`

O resultado é gravado **sempre**, mesmo quando alguma checagem falha. Usa o modo `append` (como nas Silvers) e o **mesmo log**, com `table_name = gold.fato_consumo_faturamento`. Cada execução acrescenta linhas; filtre pela `checked_at` mais recente para ver só a atual.

In [0]:
# Persistência do dq_log (modo append, igual às Silvers); sempre grava, inclusive quando há falha
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabela_dq_log)

display(
    spark.table(tabela_dq_log)
    .filter(F.col("table_name") == "gold.fato_consumo_faturamento")
    .orderBy(F.col("checked_at").desc())
    .limit(len(dq_results))
)

table_name,check_name,total_rows,failed_rows,passed,checked_at
gold.fato_consumo_faturamento,volume dos últimos 12 meses = 218.079.054 kWh (diagnóstico),2152336,0,true,2026-10-27T03:29:19.436Z
gold.fato_consumo_faturamento,volume do primeiro ano = 227.052.927 kWh (diagnóstico),2152336,0,true,2026-10-27T03:29:19.436Z
gold.fato_consumo_faturamento,soma de valor_energia = soma de silver.faturas,2152336,0,true,2026-10-27T03:29:18.782Z
gold.fato_consumo_faturamento,soma de consumo_kwh = soma de silver.leituras,2152336,0,true,2026-10-27T03:29:18.782Z
gold.fato_consumo_faturamento,dias_atraso >= 0,2152336,0,true,2026-10-27T03:29:16.377Z
gold.fato_consumo_faturamento,em_aberto = data_pagamento nula,2152336,0,true,2026-10-27T03:29:15.654Z
gold.fato_consumo_faturamento,valor_total = valor_energia + tributos,2152336,0,true,2026-10-27T03:29:14.634Z
gold.fato_consumo_faturamento,"valor_energia, tributos e valor_total >= 0",2152336,0,true,2026-10-27T03:29:13.571Z
gold.fato_consumo_faturamento,consumo_kwh >= 0,2152336,0,true,2026-10-27T03:29:12.805Z
gold.fato_consumo_faturamento,variação preenchida sempre que há consumo do ano anterior,2152336,0,true,2026-10-27T03:29:12.098Z


## 10. Gravação da Gold, só se todas as checagens passarem

Se **qualquer** checagem falhar, a tabela final **não é gravada**: o notebook imprime quais falharam e é interrompido com erro (num Workflow, isso marca a execução como falha). A tabela temporária `_stg` fica para inspeção, e a `fato_consumo_faturamento` anterior, se existir, permanece como estava.

Se todas passarem, a tabela final é gravada a partir da `_stg`, e a `_stg` é apagada.

In [0]:
# Gravação da Gold: só acontece se todas as checagens passaram
falhas = [r for r in dq_results if not r.passed]

if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada ({tabela_stg} ficou para inspeção):")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    fato.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold)
)
spark.sql(f"DROP TABLE IF EXISTS {tabela_stg}")
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {n_br(spark.table(tabela_gold).count())} linhas")

Todas as 52 checagens passaram. v_energy.gold.fato_consumo_faturamento gravada: 2.152.336 linhas


## 11. Teste de conexão e conferência com o diagnóstico

Termina com consultas que provam que a tabela se conecta e responde ao que as lideranças pediram:

1. **Conexão com o cadastro:** toda UC da fato existe em `silver.unidades_consumidoras` (e na `gold.dim_uc`, quando ela estiver pronta).
2. **Visão 360 (Camila):** uma UC com o consumo dos últimos 12 meses ao lado do mesmo mês do ano anterior, as faturas em aberto e a flag de troca de medidor.
3. **Diagnóstico (Henrique):** volume, receita e preço por período e por classe, que devem reproduzir os números do `docs/diagnostico_receita.md`.

In [0]:
fato_f = spark.table(tabela_gold)

# 1. Conexão com o cadastro (esperado: 0 UCs da fato fora do cadastro)
print("UCs da fato fora de silver.unidades_consumidoras:",
      n_br(fato_f.select("id_uc").distinct().join(spark.table(tabela_uc).select("id_uc"), "id_uc", "left_anti").count()))
try:                                                       # só roda depois que a gold.dim_uc existir
    dim_uc = spark.table(f"{gold_schema}.dim_uc").select("id_uc")
    print("UCs da fato fora de gold.dim_uc:", n_br(fato_f.select("id_uc").distinct().join(dim_uc, "id_uc", "left_anti").count()))   # esperado: 0
except Exception:
    print("gold.dim_uc ainda não existe: conexão com a dimensão fica para depois")

# GD no mês: UCs com GD conectada em 2026-06 (esperado: 3.510, se todas as conexões forem até 2026-06)
display(fato_f.filter("mes_referencia = '2026-06'").groupBy("possui_gd_no_mes").agg(F.countDistinct("id_uc").alias("ucs")))
# Faixas de atraso (a maior parte das faturas em aberto deve estar em MAIS DE 90 DIAS)
display(fato_f.groupBy("em_aberto", "faixa_atraso").count().orderBy("em_aberto", "faixa_atraso"))

# 2. Visão 360: uma UC com os 48 meses (últimos 12 meses ao lado do ano anterior)
linha_exemplo = fato_f.groupBy("id_uc").count().filter("count = 48").orderBy("id_uc").limit(1).first()
uc_exemplo = linha_exemplo["id_uc"] if linha_exemplo else fato_f.select("id_uc").limit(1).first()["id_uc"]
print("UC de exemplo:", uc_exemplo)
display(
    fato_f.filter((F.col("id_uc") == uc_exemplo) & (F.col("periodo_diagnostico") == "ULTIMOS_12M"))
    .select("mes_referencia", "consumo_kwh", "consumo_kwh_ano_anterior", "variacao_consumo_ano_anterior_pct",
            "possui_gd_no_mes", "troca_medidor", "valor_total", "bandeira_tarifaria", "status_fatura", "dias_atraso", "faixa_atraso")
    .orderBy("mes_referencia")
)
display(fato_f.filter((F.col("id_uc") == uc_exemplo) & F.col("em_aberto"))
        .select("mes_referencia", "id_fatura", "valor_total", "data_vencimento", "dias_atraso", "faixa_atraso").orderBy("mes_referencia"))

UCs da fato fora de silver.unidades_consumidoras: 0
gold.dim_uc ainda não existe: conexão com a dimensão fica para depois


possui_gd_no_mes,ucs
false,41490
true,3510


em_aberto,faixa_atraso,count
false,ATE 30 DIAS,1650796
false,EM DIA,328987
true,31 A 90 DIAS,6971
true,ATE 30 DIAS,3564
true,EM DIA,1700
true,MAIS DE 90 DIAS,160318


UC de exemplo: UC100000


mes_referencia,consumo_kwh,consumo_kwh_ano_anterior,variacao_consumo_ano_anterior_pct,possui_gd_no_mes,troca_medidor,valor_total,bandeira_tarifaria,status_fatura,dias_atraso,faixa_atraso
2025-07,160,178,-10.11,false,false,132.79,Vermelha P1,PAGA,0,EM DIA
2025-08,190,160,18.75,false,false,166.08,Vermelha P2,PAGA,21,ATE 30 DIAS
2025-09,194,210,-7.62,false,false,169.89,Vermelha P2,PAGA,22,ATE 30 DIAS
2025-10,257,272,-5.51,false,false,213.55,Vermelha P1,PAGA,18,ATE 30 DIAS
2025-11,449,311,44.37,false,false,372.75,Vermelha P1,PAGA,22,ATE 30 DIAS
2025-12,350,322,8.70,false,false,278.59,Amarela,PAGA,6,ATE 30 DIAS
2026-01,380,426,-10.80,false,false,296.48,Verde,EM ABERTO,152,MAIS DE 90 DIAS
2026-02,461,353,30.59,false,false,360.01,Verde,PAGA,5,ATE 30 DIAS
2026-03,332,407,-18.43,false,false,259.33,Verde,PAGA,18,ATE 30 DIAS
2026-04,361,363,-0.55,false,false,281.47,Verde,PAGA,16,ATE 30 DIAS


mes_referencia,id_fatura,valor_total,data_vencimento,dias_atraso,faixa_atraso
2022-11,FT000000004,234.19,2022-12-06,1302,MAIS DE 90 DIAS
2023-06,FT000000011,141.89,2023-07-03,1093,MAIS DE 90 DIAS
2023-10,FT000000015,196.16,2023-11-05,968,MAIS DE 90 DIAS
2024-10,FT000000027,234.21,2024-11-06,601,MAIS DE 90 DIAS
2026-01,FT000000042,296.48,2026-01-29,152,MAIS DE 90 DIAS


In [0]:
# 3. Diagnóstico: reproduz os números do docs/diagnostico_receita.md
per = (fato_f.filter(F.col("periodo_diagnostico") != "OUTRO")
       .groupBy("periodo_diagnostico")
       .agg(F.sum("consumo_kwh").alias("volume_kwh"),
            F.sum("valor_energia").alias("receita_energia"),
            F.sum("valor_total").alias("receita_total"),
            F.countDistinct("id_uc").alias("ucs"))
       .withColumn("preco_medio", F.round(F.col("receita_energia") / F.col("volume_kwh"), 4)))
display(per.orderBy("periodo_diagnostico"))
# esperado: volume 227.052.927 -> 218.079.054 (-3,95%); receita de energia R$ 117,83 mi -> R$ 127,26 mi (+8,00%);
#           receita total R$ 157,11 mi -> R$ 169,68 mi (+8,00%); preço médio R$ 0,5190 -> R$ 0,5836 por kWh (+12,45%);
#           UCs com fatura: 44.741 -> 45.000

# Por classe (esperado, volume P1 contra P0: comercial -10,2%, residencial -5,4%, industrial +0,6%, rural +2,5%)
cl = (fato_f.filter(F.col("periodo_diagnostico") != "OUTRO")
      .groupBy("classe")
      .pivot("periodo_diagnostico", ["PRIMEIRO_ANO", "ULTIMOS_12M"])
      .agg(F.sum("consumo_kwh"))
      .withColumn("variacao_volume_pct", F.round((F.col("ULTIMOS_12M") / F.col("PRIMEIRO_ANO") - 1) * 100, 1)))
display(cl.orderBy("variacao_volume_pct"))

periodo_diagnostico,volume_kwh,receita_energia,receita_total,ucs,preco_medio
PRIMEIRO_ANO,227052927,117830394.38,157107191.43,44741,0.5190
ULTIMOS_12M,218079054,127262595.65,169683458.59,45000,0.5836


classe,PRIMEIRO_ANO,ULTIMOS_12M,variacao_volume_pct
COMERCIAL,51834211,46571036,-10.2
RESIDENCIAL,81686716,77307032,-5.4
INDUSTRIAL,87177795,87690540,0.6
PODER PUBLICO,4111425,4211184,2.4
RURAL,2242780,2299262,2.5
